In [1]:
import os,sys
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/V-Max_Hackerz/
sys.path.insert(0,'/content/drive/Mydrive/V-Max_Hackerz/notebook')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/V-Max_Hackerz


In [ ]:
!find . -name "normalization.py"

./code/business_entity_resolution/src/business_entity_resolution/preprocessing/normalization.py


In [ ]:
!touch "./code/business_entity_resolution/src/business_entity_resolution/__init__.py"
!touch "./code/business_entity_resolution/src/business_entity_resolution/preprocessing/__init__.py"

In [2]:
import sys
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/V-Max_Hackerz")

TRAIN_DIR = PROJECT_ROOT / "student_resource"/"dataset" / "train"
TEST_DIR = PROJECT_ROOT / "student_resource"/"dataset" / "test"
OUTPUT_DIR = PROJECT_ROOT / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find normalization.py automatically anywhere in your Drive project
normalization_file = next(Path("/content/drive/MyDrive").rglob("normalization.py"))

# The src directory is two levels above normalization.py (src / business_entity_resolution / preprocessing / normalization.py)
SRC_DIR = normalization_file.parents[2]

sys.path.insert(0, str(SRC_DIR))

from business_entity_resolution.preprocessing.normalization import (
    normalize_name,
    normalize_address,
)

print("Project:", PROJECT_ROOT)
print("Output:", OUTPUT_DIR)

Project: /content/drive/MyDrive/V-Max_Hackerz
Output: /content/drive/MyDrive/V-Max_Hackerz/output


In [3]:
s1 = pd.read_csv(
    TEST_DIR / "test_source1.tsv",
    sep="\t"
)

s2 = pd.read_csv(
    TEST_DIR / "test_source2.tsv",
    sep="\t"
)

s3 = pd.read_csv(
    TEST_DIR / "test_source3.tsv",
    sep="\t"
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)

S1: (1732544, 4)
S2: (3903296, 4)
S3: (4107443, 4)


In [4]:
candidates = pd.concat(
    [
        s2[["entity_id", "business_name", "business_address", "country"]],
        s3[["entity_id", "business_name", "business_address", "country"]],
    ],
    ignore_index=True
)

print(candidates.shape)

(8010739, 4)


In [5]:
name_index = defaultdict(list)

for row in candidates.itertuples(index=False):

    name = normalize_name(row.business_name)

    if not name:
        continue

    key = (row.country, name)

    name_index[key].append(row.entity_id)

print("Name index keys:", len(name_index))

Name index keys: 6208326


In [6]:
address_index = defaultdict(list)

for row in candidates.itertuples(index=False):

    address = normalize_address(row.business_address)

    if not address:
        continue

    key = (row.country, address)

    address_index[key].append(row.entity_id)

print("Address index keys:", len(address_index))

Address index keys: 6772627


In [7]:
ADDRESS_STOPWORDS = {
    "road","street","avenue","boulevard","drive","lane","highway",
    "parkway","court","place","trail","circle","suite","apartment",
    "floor","flat","unit","block","plot","sector","nagar","colony",
    "near","opposite","opp","main","city","county","north","south",
    "east","west","the","and","for","null"
}

def informative_address_tokens(text):

    if pd.isna(text) or not str(text).strip():
        return []

    text = normalize_address(text)

    tokens = text.split()

    return [
        token
        for token in tokens
        if len(token) >= 3
        and token not in ADDRESS_STOPWORDS
    ]

In [8]:
pair_counter = Counter()

for row in candidates.itertuples(index=False):

    tokens = set(
        informative_address_tokens(row.business_address)
    )

    if len(tokens) < 2:
        continue

    for token1, token2 in combinations(sorted(tokens), 2):

        key = (row.country, token1, token2)

        pair_counter[key] += 1

print("Total address-pair keys:", len(pair_counter))

Total address-pair keys: 15219551


In [9]:
PAIR_THRESHOLD = 50

address_pair_index = defaultdict(list)

for row in candidates.itertuples(index=False):

    tokens = set(
        informative_address_tokens(row.business_address)
    )

    if len(tokens) < 2:
        continue

    for token1, token2 in combinations(sorted(tokens), 2):

        key = (row.country, token1, token2)

        if pair_counter.get(key, 0) <= PAIR_THRESHOLD:

            address_pair_index[key].append(
                row.entity_id
            )

print(
    "Selective address-pair keys:",
    len(address_pair_index)
)

Selective address-pair keys: 15035368


In [10]:
NAME_STOPWORDS = {
    "private","limited","corporation","company","incorporated",
    "llc","llp","pllc","corp","inc","co",
    "group","holding","holdings","services","service",
    "solutions","enterprises","international","global",
    "partners","associates","consultancy","consultants",
    "and","india"
}

def informative_name_tokens(text):

    if pd.isna(text) or not str(text).strip():
        return []

    text = normalize_name(text)

    tokens = text.split()

    return [
        token
        for token in tokens
        if len(token) >= 3
        and token not in NAME_STOPWORDS
    ]

name_token_index = defaultdict(list)

for row in candidates.itertuples(index=False):

    tokens = set(
        informative_name_tokens(row.business_name)
    )

    for token in tokens:

        key = (row.country, token)

        name_token_index[key].append(
            row.entity_id
        )

print(
    "Name-token keys:",
    len(name_token_index)
)

largest_name_blocks = sorted(
    (
        (key, len(ids))
        for key, ids in name_token_index.items()
    ),
    key=lambda x: x[1],
    reverse=True
)

largest_name_blocks[:20]

Name-token keys: 1094717


[(('France', 'sarl'), 229169),
 (('France', 'sas'), 161110),
 (('US', 'center'), 139637),
 (('India', 'center'), 109873),
 (('US', 'com'), 108917),
 (('India', 'com'), 108715),
 (('France', 'france'), 107723),
 (('India', 'industries'), 101063),
 (('India', 'ventures'), 91153),
 (('India', 'public'), 89549),
 (('India', 'exports'), 82249),
 (('US', 'care'), 81203),
 (('France', 'club'), 79722),
 (('France', 'eurl'), 67139),
 (('India', 'infratech'), 67121),
 (('India', 'overseas'), 66798),
 (('India', 'brothers'), 65439),
 (('India', 'trading'), 65413),
 (('France', 'amicale'), 52426),
 (('India', 'technologies'), 50812)]

In [11]:
NAME_TOKEN_THRESHOLD = 500

name_token_index_selective = defaultdict(list)

for key, ids in name_token_index.items():

    if len(ids) <= NAME_TOKEN_THRESHOLD:
        name_token_index_selective[key] = ids

print(
    "All name-token keys:",
    len(name_token_index)
)

print(
    "Selective name-token keys:",
    len(name_token_index_selective)
)

All name-token keys: 1094717
Selective name-token keys: 1091254


In [13]:
largest_selective_name_blocks = sorted(
    (
        (key, len(ids))
        for key, ids in name_token_index_selective.items()
    ),
    key=lambda x: x[1],
    reverse=True
)

largest_selective_name_blocks[:20]

[(('India', 'କନଷ'), 500),
 (('India', 'ରକସନ'), 500),
 (('France', '3eme'), 500),
 (('France', 'jules'), 500),
 (('India', 'asset'), 500),
 (('India', 'ácademy'), 500),
 (('India', 'share'), 500),
 (('India', 'office'), 500),
 (('India', 'hóspitality'), 500),
 (('US', 'aguirre'), 500),
 (('India', 'scientific'), 500),
 (('US', 'reeves'), 499),
 (('India', 'vista'), 499),
 (('India', 'éxports'), 498),
 (('US', 'vaughn'), 498),
 (('India', 'sant'), 498),
 (('India', 'bridge'), 498),
 (('India', 'developersprivate'), 498),
 (('US', 'simon'), 498),
 (('India', 'fast'), 498)]

In [14]:
print("Selective name-token keys:", len(name_token_index_selective))
largest_selective_name_blocks[:10]

Selective name-token keys: 1091254


[(('India', 'କନଷ'), 500),
 (('India', 'ରକସନ'), 500),
 (('France', '3eme'), 500),
 (('France', 'jules'), 500),
 (('India', 'asset'), 500),
 (('India', 'ácademy'), 500),
 (('India', 'share'), 500),
 (('India', 'office'), 500),
 (('India', 'hóspitality'), 500),
 (('US', 'aguirre'), 500)]

In [15]:
def get_name_candidates(country, business_name):
    name = normalize_name(business_name)

    if not name:
        return set()

    return set(
        name_index.get((country, name), [])
    )


def get_address_candidates(country, business_address):
    address = normalize_address(business_address)

    if not address:
        return set()

    return set(
        address_index.get((country, address), [])
    )


def get_name_token_candidates(country, business_name):
    tokens = set(
        informative_name_tokens(business_name)
    )

    result = set()

    for token in tokens:
        key = (country, token)

        ids = name_token_index_selective.get(key)

        if ids:
            result.update(ids)

    return result


def get_address_pair_candidates(country, business_address):
    tokens = set(
        informative_address_tokens(business_address)
    )

    if len(tokens) < 2:
        return set()

    result = set()

    for token1, token2 in combinations(sorted(tokens), 2):

        key = (country, token1, token2)

        ids = address_pair_index.get(key)

        if ids:
            result.update(ids)

    return result

In [16]:
for row in s1.head(5).itertuples(index=False):

    name_cands = get_name_candidates(
        row.country,
        row.business_name
    )

    address_cands = get_address_candidates(
        row.country,
        row.business_address
    )

    name_token_cands = get_name_token_candidates(
        row.country,
        row.business_name
    )

    address_pair_cands = get_address_pair_candidates(
        row.country,
        row.business_address
    )

    all_candidates = (
        name_cands
        | address_cands
        | name_token_cands
        | address_pair_cands
    )

    print(
        row.entity_id,
        "→",
        len(all_candidates),
        "candidates"
    )

S1-714132312 → 66 candidates
S1-106407869 → 19 candidates
S1-156285671 → 45 candidates
S1-689823050 → 134 candidates
S1-921369899 → 72 candidates


In [18]:
CANDIDATE_OUTPUT = OUTPUT_DIR / "candidate_pairs.tsv"

CHUNK_SIZE = 10_000

# Start fresh
with open(CANDIDATE_OUTPUT, "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")

total_candidates = 0
rows_processed = 0
zero_candidates = 0
max_candidates = 0

for start in range(0, len(s1), CHUNK_SIZE):

    end = min(start + CHUNK_SIZE, len(s1))
    chunk = s1.iloc[start:end]

    rows = []

    for row in chunk.itertuples(index=False):

        candidates_for_s1 = set()

        # Block A — exact normalized name
        candidates_for_s1.update(
            get_name_candidates(
                row.country,
                row.business_name
            )
        )

        # Block B — exact normalized address
        candidates_for_s1.update(
            get_address_candidates(
                row.country,
                row.business_address
            )
        )

        # Block C — informative name token, frequency <= 500
        candidates_for_s1.update(
            get_name_token_candidates(
                row.country,
                row.business_name
            )
        )

        # Block D — address token pair, frequency <= 50
        candidates_for_s1.update(
            get_address_pair_candidates(
                row.country,
                row.business_address
            )
        )

        candidate_ids = sorted(candidates_for_s1)

        candidate_count = len(candidate_ids)

        total_candidates += candidate_count
        rows_processed += 1

        if candidate_count == 0:
            zero_candidates += 1

        max_candidates = max(
            max_candidates,
            candidate_count
        )

        rows.append(
            (
                row.entity_id,
                ",".join(candidate_ids)
            )
        )

    # Write chunk
    with open(
        CANDIDATE_OUTPUT,
        "a",
        encoding="utf-8"
    ) as f:

        for entity_id, candidate_ids in rows:
            f.write(
                f"{entity_id}\t{candidate_ids}\n"
            )

    print(
        f"Processed {end:,}/{len(s1):,} "
        f"({end / len(s1) * 100:.1f}%) | "
        f"Chunk candidates: "
        f"{sum(len(x[1].split(',')) if x[1] else 0 for x in rows):,}"
    )

print("\n===== FINAL STATISTICS =====")

print("S1 records:", rows_processed)
print("Total candidate links:", total_candidates)

print(
    "Average candidates/S1:",
    total_candidates / rows_processed
)

print(
    "S1 with zero candidates:",
    zero_candidates
)

print(
    "Maximum candidates for one S1:",
    max_candidates
)

print(
    "Candidate file:",
    CANDIDATE_OUTPUT
)

Processed 10,000/1,732,544 (0.6%) | Chunk candidates: 1,170,808
Processed 20,000/1,732,544 (1.2%) | Chunk candidates: 1,203,757
Processed 30,000/1,732,544 (1.7%) | Chunk candidates: 1,198,087
Processed 40,000/1,732,544 (2.3%) | Chunk candidates: 1,185,772
Processed 50,000/1,732,544 (2.9%) | Chunk candidates: 1,181,181
Processed 60,000/1,732,544 (3.5%) | Chunk candidates: 1,196,279
Processed 70,000/1,732,544 (4.0%) | Chunk candidates: 1,201,145
Processed 80,000/1,732,544 (4.6%) | Chunk candidates: 1,186,569
Processed 90,000/1,732,544 (5.2%) | Chunk candidates: 1,207,695
Processed 100,000/1,732,544 (5.8%) | Chunk candidates: 1,201,529
Processed 110,000/1,732,544 (6.3%) | Chunk candidates: 1,194,270
Processed 120,000/1,732,544 (6.9%) | Chunk candidates: 1,192,169
Processed 130,000/1,732,544 (7.5%) | Chunk candidates: 1,172,593
Processed 140,000/1,732,544 (8.1%) | Chunk candidates: 1,199,905
Processed 150,000/1,732,544 (8.7%) | Chunk candidates: 1,196,853
Processed 160,000/1,732,544 (9.2%)

In [19]:
candidate_file = OUTPUT_DIR / "candidate_pairs.tsv"

# Read only the output file
cp = pd.read_csv(
    candidate_file,
    sep="\t",
    dtype=str,
    keep_default_na=False
)

print("Shape:", cp.shape)
print("Columns:", cp.columns.tolist())

print("\nDuplicate S1 IDs:")
print(cp["source1_entity_id"].duplicated().sum())

print("\nEmpty candidate lists:")
print(
    (cp["candidate_entity_ids"] == "").sum()
)

print("\nSample:")
display(cp.head())

Shape: (1732544, 2)
Columns: ['source1_entity_id', 'candidate_entity_ids']

Duplicate S1 IDs:
0

Empty candidate lists:
10035

Sample:


,source1_entity_id,candidate_entity_ids
0,S1-714132312,"S2-170689739,S2-185496332,S2-187020300,S2-1938..."
1,S1-106407869,"S2-104974637,S2-114144622,S2-181714769,S2-3529..."
2,S1-156285671,"S2-160966771,S2-172049742,S2-181576625,S2-2104..."
3,S1-689823050,"S2-116479052,S2-131379034,S2-133253656,S2-1521..."
4,S1-921369899,"S2-106862520,S2-126006680,S2-129925079,S2-1815..."


In [20]:
invalid_prefix = 0
duplicate_ids = 0

for ids in cp["candidate_entity_ids"]:

    if not ids:
        continue

    id_list = ids.split(",")

    if len(id_list) != len(set(id_list)):
        duplicate_ids += 1

    invalid_prefix += sum(
        not (x.startswith("S2-") or x.startswith("S3-"))
        for x in id_list
    )

print("Rows containing duplicate candidate IDs:", duplicate_ids)
print("Invalid candidate IDs:", invalid_prefix)

Rows containing duplicate candidate IDs: 0
Invalid candidate IDs: 0


In [21]:
print(
    "Expected S1:",
    len(s1)
)

print(
    "Candidate rows:",
    len(cp)
)

print(
    "Missing S1:",
    len(set(s1["entity_id"]) - set(cp["source1_entity_id"]))
)

print(
    "Extra S1:",
    len(set(cp["source1_entity_id"]) - set(s1["entity_id"]))
)

Expected S1: 1732544
Candidate rows: 1732544
Missing S1: 0
Extra S1: 0


In [22]:
candidate_counts = cp["candidate_entity_ids"].apply(
    lambda x: 0 if not x else len(x.split(","))
)

print(candidate_counts.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\nZero candidates:", (candidate_counts == 0).sum())
print("Total candidate links:", candidate_counts.sum())

count    1.732544e+06
mean     1.195167e+02
std      1.352770e+02
min      0.000000e+00
50%      7.200000e+01
90%      3.100000e+02
95%      4.220000e+02
99%      5.920000e+02
max      1.949000e+03
Name: candidate_entity_ids, dtype: float64

Zero candidates: 10035
Total candidate links: 207067922
